# Cross-check: the pipeline vs pandas and statsmodels

`src/cx_driver_analysis.py` is written in the Python standard library (it even
fits its own logistic regression), so it reruns anywhere with no install.
This notebook recomputes the headline results **with pandas and statsmodels**
and asserts they agree with `reports/cx_driver_metrics.json`. If an `assert`
fails, the notebook (and CI) fails.

| Result | Recomputed here with |
| --- | --- |
| Satisfied rate + Wilson 95% CI | `statsmodels.stats.proportion.proportion_confint` |
| NPS proxy | pandas `value_counts` |
| Late vs on-time delivery (z-test, gap CI) | `proportions_ztest`, `confint_proportions_2indep` |
| Multivariate drivers (odds ratios) | `statsmodels.formula.api.logit` |

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import confint_proportions_2indep, proportion_confint, proportions_ztest

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
m = json.loads((ROOT / "reports/cx_driver_metrics.json").read_text())
df = pd.read_csv(ROOT / "data/olist_reviews.csv")
print(f"{len(df):,} real Olist reviews")

95,639 real Olist reviews


## 1. Satisfied rate (score 4-5) with a Wilson interval

In [2]:
sat, n = int(df.satisfied.sum()), len(df)
lo, hi = proportion_confint(sat, n, alpha=0.05, method="wilson")
assert abs(sat / n - m["overall_satisfied_rate"]) < 1e-4
assert np.allclose([lo, hi], m["overall_satisfied_ci_95"], atol=1e-4)
print(f"satisfied {sat / n:.2%}  95% CI [{lo:.2%}, {hi:.2%}]")

satisfied 78.97%  95% CI [78.71%, 79.23%]


## 2. NPS proxy (5 = promoter, 4 = passive, ≤3 = detractor)

In [3]:
bands = df.nps_band.value_counts(normalize=True)
nps = (bands["promoter"] - bands["detractor"]) * 100
assert abs(nps - m["nps"]["overall"]["nps"]) < 0.05
bands.mul(100).round(1).rename("% of reviews").to_frame()

,% of reviews
nps_band,
promoter,59.3
detractor,21.0
passive,19.7


## 3. Late vs on-time delivery — two-proportion z-test

In [4]:
g = df.groupby("late_delivery").satisfied.agg(["sum", "count"])
on_time, late = g.loc[0], g.loc[1]
z, p = proportions_ztest([on_time["sum"], late["sum"]], [on_time["count"], late["count"]])
lo, hi = confint_proportions_2indep(on_time["sum"], on_time["count"], late["sum"], late["count"],
                                   method="wald", compare="diff")
c = m["delivery_sla_cohort"]
assert abs(z - c["z"]) < 1e-2 and np.allclose([lo, hi], c["rate_gap_ci"], atol=1e-4)
pd.DataFrame({"satisfied_rate": g["sum"] / g["count"], "reviews": g["count"]},
             index=pd.Index([0, 1], name="late_delivery")).round(4)

,satisfied_rate,reviews
late_delivery,,
0,0.8270,89277
1,0.2671,6362


## 4. Multivariate logistic regression — statsmodels

Same design as the pipeline: late flag, standardised delivery days and freight
ratio, dummies for the top states and categories (the rest pooled as baseline).
The pipeline adds a tiny ridge penalty (λ = 1) for numerical stability; on
95k rows it moves the odds ratios by well under 1%, which is the tolerance here.

In [5]:
names = [c["feature"] for c in m["multivariate_drivers"]["coefficients"]]
X = pd.DataFrame({"late_delivery": df.late_delivery})
for col, src in (("delivery_days_z", "delivery_days"), ("freight_ratio_z", "freight_ratio")):
    X[col] = (df[src] - df[src].mean()) / df[src].std(ddof=0)
for nm in names:
    if nm.startswith("state="):
        X[nm] = (df.customer_state == nm[6:]).astype(int)
    elif nm.startswith("category="):
        X[nm] = (df.product_category == nm[9:]).astype(int)
X.columns = [c.replace("=", "_") for c in X.columns]
data = X.assign(satisfied=df.satisfied)

fit = smf.logit("satisfied ~ " + " + ".join(X.columns), data=data).fit(disp=False)
ours = pd.Series({c["feature"].replace("=", "_"): c["odds_ratio"] for c in m["multivariate_drivers"]["coefficients"]})
theirs = np.exp(fit.params).rename({"Intercept": "intercept"})
rel_diff = (theirs / ours[theirs.index] - 1).abs()
assert rel_diff.max() < 0.01, rel_diff.sort_values().tail()

table = pd.DataFrame({"pipeline_OR": ours[theirs.index], "statsmodels_OR": theirs,
                      "statsmodels_p": fit.pvalues.rename({"Intercept": "intercept"})})
table.drop("intercept").round(4).sort_values("statsmodels_OR")

,pipeline_OR,statsmodels_OR,statsmodels_p
late_delivery,0.1841,0.1837,0.0000
delivery_days_z,0.6507,0.6510,0.0000
category_bed_bath_table,0.7379,0.7377,0.0000
state_RJ,0.7904,0.7902,0.0000
state_SP,0.7941,0.7940,0.0000
category_telephony,0.8064,0.8060,0.0000
category_furniture_decor,0.8228,0.8225,0.0000
state_MG,0.8465,0.8463,0.0000
category_watches_gifts,0.8642,0.8639,0.0001
category_computers_accessories,0.8643,0.8640,0.0000


**All assertions passed.** Late delivery cuts the odds of a satisfied review
by about 80% (odds ratio ≈ 0.18) even after controlling for state, category, delivery time and freight
— the same answer from the hand-written estimator and from statsmodels.